# OLX Ko'chmas mulk — Malumot Tozalash jaroyoni  
Kodni tepadan paga qarab yurg'azing. Oxirida sizda tayor `cleaned.csv` fileli modeling uchun tayor bo'ladi.

In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [5]:
df = pd.read_csv('/Users/javohireshonov/Desktop/Study/Projects/Tashkent apartment analysis/data/raw/olx_apartments.csv')
print(f'Loaded: {df.shape[0]:,} rows, {df.shape[1]} columns')

Loaded: 18,986 rows, 28 columns


In [6]:
df.isna().sum()

listing_id             0
seller_type            1
housing_type        5956
region                 1
district               1
rooms                  1
living_area_m2     12862
kitchen_area_m2    14775
total_area_m2          4
floor                  2
total_floors           1
building_type       4488
layout              6378
build_year         13785
ceiling_height     12486
bathroom            5057
furnished              3
renovation          3370
commission             1
amenities           7045
nearby              6523
negotiable             0
price                 40
currency               0
published_date         1
description            1
date_scraped           0
url                    0
dtype: int64

In [7]:
df.head()

,listing_id,seller_type,housing_type,region,district,rooms,living_area_m2,kitchen_area_m2,total_area_m2,floor,...,commission,amenities,nearby,negotiable,price,currency,published_date,description,date_scraped,url
0,4jYx9,private,resale,Tashkent Region,Chilanzar District,1.0,20.0,6.0,29.0,1.0,...,0.0,"Internet, Refrigerator, TV, Air Conditioning, ...","Hospital, Clinic, Playground, Kindergarten, Bu...",1,60000.0,USD,01/05/2026,Описание Уй локацияси зур жойда жойлашган. Мир...,2026-05-02,https://www.olx.uz/d/obyavlenie/1-hona-metro-o...
1,4myoT,private,NaN,Tashkent Region,Yunusabad District,2.0,80.0,NaN,80.0,6.0,...,1.0,"Internet, Refrigerator, TV, Air Conditioning, ...","Hospital, Clinic, Playground, Kindergarten, Bu...",0,7865000.0,UZS,30/04/2026,Описание Сдается в аренду 2/6/13 Юнусабад 13 к...,2026-05-02,https://www.olx.uz/d/obyavlenie/sdaetsya-v-are...
2,4lvjL,business,new building,Tashkent Region,Yunusabad District,3.0,NaN,NaN,80.0,6.0,...,0.0,"Internet, Refrigerator, TV, Air Conditioning, ...","Hospital, Clinic, School, Playground, Kinderga...",0,220000.0,USD,29/04/2026,Описание Продается квартира !!! 3/6/9 80 кВ По...,2026-05-02,https://www.olx.uz/d/obyavlenie/srochno-prodae...
3,4aiOy,private,NaN,Tashkent Region,Shaykhantakhur District,3.0,67.5,NaN,67.5,5.0,...,0.0,"Internet, Refrigerator, TV, Air Conditioning, ...",NaN,0,1400.0,USD,02/05/2026,"Описание Квартира в новостройке Tashkent City,...",2026-05-02,https://www.olx.uz/d/obyavlenie/sdaetsya-v-are...
4,4mAGS,business,resale,Tashkent Region,Yashnabad District,3.0,50.0,NaN,78.0,1.0,...,1.0,"Internet, Telephone, Refrigerator, TV, Air Con...","Hospital, Clinic, Playground, Kindergarten, Bu...",0,120000.0,USD,02/05/2026,Описание Продается своя квартира 3/1/4 Кирпичн...,2026-05-02,https://www.olx.uz/d/obyavlenie/kvartira-3-1-4...


## A. Malumotna uchun shartnoma

# Ma'lumotlar lug'ati — OLX.uz kvartiralar dataseti
 
> **Hajmi:** 18 000+ qator × 28 ustun  
> **Manba:** OLX.uz — kvartiralar bo'limi (`/nedvizhimost/kvartiry/`)  
> **Skraping sanasi:** 2026-04-15 - hozir
 
---
 
| Ustun | Ma'nosi | Turi | Mumkin qiymatlar / diapazon | O'lchov birligi | Izohlar |
| --- | --- | --- | --- | --- | --- |
| `listing_id` | E'lon unikal identifikatori | string | `'4jYx9'` kabi 5 belgili alfanumerik kod | — | OLX tomonidan avtomatik beriladi; asosiy kalit (primary key) sifatida ishlatiladi |
| `seller_type` | Sotuvchi turi | kategoriya | `private`, `business` | — | `private` — uy egasi o'zi; `business` — agentlik yoki firma |
| `housing_type` | Uy-joy turi | kategoriya | `resale`, `new building` | — | 2 834 ta qiymat yo'q; ko'pchilik `resale` |
| `region` | Viloyat | kategoriya | `Tashkent Region` va boshqa viloyatlar | — | Hozirgi ma'lumotlarda asosan Toshkent viloyati |
| `district` | Tuman / mahalla | kategoriya | `Chilanzar District`, `Yunusabad District` va boshqalar | — | Narx prognozi uchun eng muhim ustunlardan biri |
| `rooms` | Xonalar soni | raqam | `1` — `6+` | dona | Butun son kutiladi; modelda kategoriya sifatida ham ishlatish mumkin |
| `living_area_m2` | Yashash maydoni | raqam | taxminan `10` — `200` | m² | 5 777 ta qiymat yo'q (66%); mediana bilan to'ldirish + `missing` flagi tavsiya etiladi |
| `kitchen_area_m2` | Oshxona maydoni | raqam | taxminan `4` — `30` | m² | 6 656 ta qiymat yo'q (76%); yuqori yo'qotish darajasi |
| `total_area_m2` | Umumiy maydon | raqam | taxminan `15` — `500` | m² | Faqat 1 ta qiymat yo'q; narx prognozi uchun asosiy ustun |
| `floor` | Qavat | raqam | `1` — `30+` | qavat | 1 ta qiymat yo'q; mediana bilan to'ldirish yetarli |
| `total_floors` | Binodagi jami qavatlar soni | raqam | `2` — `30+` | qavat | Qiymat yo'q emas; `floor / total_floors` nisbati yaratish tavsiya etiladi |
| `building_type` | Bino turi | kategoriya | `panel`, `brick`, `monolith` | — | 2 029 ta qiymat yo'q; eng ko'p uchraydigan qiymat bilan to'ldirish |
| `layout` | Xonalar tartibi | kategoriya | `adjacent`, `separate`, `Смежно-раздельная` | — | Aralash til (rus/ingliz) bor — tozalash kerak; 2 840 ta qiymat yo'q |
| `build_year` | Qurilgan yil | raqam | `1950` — `2026` | yil | 6 197 ta qiymat yo'q; `building_age = 2026 - build_year` ustuni hosil qilish tavsiya etiladi |
| `ceiling_height` | Shift balandligi | raqam | `2.5` — `4.0` | metr | 5 683 ta qiymat yo'q; mediana bilan to'ldirish |
| `bathroom` | Hammom turi | kategoriya | `separate`, `combined` | — | 2 312 ta qiymat yo'q; eng ko'p uchraydigan qiymat bilan to'ldirish |
| `furnished` | Mebel mavjudligi | ikkilik (binary) | `1` — mebel bor, `0` — yo'q | — | Faqat 2 ta qiymat yo'q |
| `renovation` | Ta'mirlash holati | kategoriya | `euro renovation`, `average condition`, `designer renovation` | — | 1 538 ta qiymat yo'q; narxga sezilarli ta'sir ko'rsatadi |
| `commission` | Vositachi komissiyasi | ikkilik (binary) | `0` — yo'q, `1` — bor | — | Qiymat yo'q emas; `1` bo'lsa agentlik orqali sotiladi |
| `amenities` | Jihozlar ro'yxati | vergul bilan ajratilgan string | `Internet`, `TV`, `Air Conditioning`, `Balcony`, `Washing Machine` va boshqalar | — | 3 187 ta qiymat yo'q = jihozlar ko'rsatilmagan; multi-hot encoding tavsiya etiladi |
| `nearby` | Yaqin atrofdagi ob'ektlar | vergul bilan ajratilgan string | `School`, `Bus Stop`, `Park`, `Hospital`, `Supermarket` va boshqalar | — | 2 874 ta qiymat yo'q; multi-hot encoding tavsiya etiladi |
| `negotiable` | Narx kelishiladi | ikkilik (binary) | `1` — ha, `0` — yo'q | — | Qiymat yo'q emas |
| `price` | Narx | raqam | `500` — `2 000 000` (USDga o'tkazilgandan keyin) | USD yoki UZS | 13 ta qiymat yo'q — bu qatorlarni o'chirish kerak; avval valyutani normallashtirish shart |
| `currency` | Valyuta | kategoriya | `USD`, `UZS` | — | Modelga kirishdan oldin hammasi USDga aylantiriladi (1 USD ≈ 12 700 UZS) |
| `published_date` | E'lon joylashtirilgan sana | sana | `DD/MM/YYYY` formatida | sana | Modelda `days_since_posted` ustuniga aylantirish mumkin |
| `description` | E'lon matni | erkin matn | Rus va o'zbek tillarida | — | Modelga kiritilmaydi; NLP tahlili uchun alohida ishlatish mumkin |
| `date_scraped` | Skraping sanasi | sana | `YYYY-MM-DD` formatida | sana | Barcha qatorlarda `2026-05-02`; modelda ishlatilmaydi |
| `url` | E'lon havolasi | string | `https://www.olx.uz/d/obyavlenie/...` | — | Modelda ishlatilmaydi; manba sifatida saqlab qo'yish tavsiya etiladi |
 
---
 
## Tozalash bo'yicha qisqacha yo'riqnoma
 
| Muammo | Ta'sirlangan ustunlar | Yechim |
| --- | --- | --- |
| Maqsadli qiymat yo'q | `price` (13 ta) | Qatorlarni o'chirish |
| Yuqori yo'qotish (>50%) | `living_area_m2`, `kitchen_area_m2`, `build_year`, `ceiling_height` | Mediana bilan to'ldirish + `_missing` flagi |
| O'rtacha yo'qotish | `housing_type`, `building_type`, `layout`, `bathroom`, `renovation` | Eng ko'p uchraydigan qiymat (mode) bilan to'ldirish |
| Aralash valyuta | `price`, `currency` | Hammasi USDga o'tkazish |
| Aralash til | `layout` | `Смежно-раздельная` → `mixed` deb standartlashtirish |
| Vergul bilan ajratilgan qiymatlar | `amenities`, `nearby` | Multi-hot encoding (har bir teg — alohida ustun) |

## B. Ma'lumotlarni qabul qilish va dastlabki tekshiruvlar

### B.1. Asosiy qabul qilish ro'yxati:

In [20]:
print(f'File nomi: olx_apartments.csv')
print(f'Qatorlar soni: {df.shape[0]}')
print(f'Ustunlar soni: {df.shape[1]}')
print(f"Ustun nomlari malumot ustuniga o'xshashmimi: (Ha/yoq): Ha")

File nomi: olx_apartments.csv
Qatorlar soni: 18986
Ustunlar soni: 28
Ustun nomlari malumot ustuniga o'xshashmimi: (Ha/yoq): Ha


### B.2. Noyoblik + dublikatlar (Dunyo talabi)

In [18]:
before = len(df)
after = df.duplicated().sum()
print(f"Dublikatlar soni: {after}")
print(f'Noyoblik soni: {before-after}')
df_cleaned = df.drop_duplicates()

Dublikatlar soni: 681
Noyoblik soni: 18305


In [19]:
df_cleaned.head()

,listing_id,seller_type,housing_type,region,district,rooms,living_area_m2,kitchen_area_m2,total_area_m2,floor,...,commission,amenities,nearby,negotiable,price,currency,published_date,description,date_scraped,url
0,4jYx9,private,resale,Tashkent Region,Chilanzar District,1.0,20.0,6.0,29.0,1.0,...,0.0,"Internet, Refrigerator, TV, Air Conditioning, ...","Hospital, Clinic, Playground, Kindergarten, Bu...",1,60000.0,USD,01/05/2026,Описание Уй локацияси зур жойда жойлашган. Мир...,2026-05-02,https://www.olx.uz/d/obyavlenie/1-hona-metro-o...
1,4myoT,private,NaN,Tashkent Region,Yunusabad District,2.0,80.0,NaN,80.0,6.0,...,1.0,"Internet, Refrigerator, TV, Air Conditioning, ...","Hospital, Clinic, Playground, Kindergarten, Bu...",0,7865000.0,UZS,30/04/2026,Описание Сдается в аренду 2/6/13 Юнусабад 13 к...,2026-05-02,https://www.olx.uz/d/obyavlenie/sdaetsya-v-are...
2,4lvjL,business,new building,Tashkent Region,Yunusabad District,3.0,NaN,NaN,80.0,6.0,...,0.0,"Internet, Refrigerator, TV, Air Conditioning, ...","Hospital, Clinic, School, Playground, Kinderga...",0,220000.0,USD,29/04/2026,Описание Продается квартира !!! 3/6/9 80 кВ По...,2026-05-02,https://www.olx.uz/d/obyavlenie/srochno-prodae...
3,4aiOy,private,NaN,Tashkent Region,Shaykhantakhur District,3.0,67.5,NaN,67.5,5.0,...,0.0,"Internet, Refrigerator, TV, Air Conditioning, ...",NaN,0,1400.0,USD,02/05/2026,"Описание Квартира в новостройке Tashkent City,...",2026-05-02,https://www.olx.uz/d/obyavlenie/sdaetsya-v-are...
4,4mAGS,business,resale,Tashkent Region,Yashnabad District,3.0,50.0,NaN,78.0,1.0,...,1.0,"Internet, Telephone, Refrigerator, TV, Air Con...","Hospital, Clinic, Playground, Kindergarten, Bu...",0,120000.0,USD,02/05/2026,Описание Продается своя квартира 3/1/4 Кирпичн...,2026-05-02,https://www.olx.uz/d/obyavlenie/kvartira-3-1-4...
